## Imports

In [ ]:
import os
import random

import cv2
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, WeightedRandomSampler, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import albumentations as A
from albumentations.pytorch import ToTensorV2

from shared.early_stopping import EarlyStopping
from shared.mat_reader import MatReader
from shared.constants import CLASS_NAMES, DEVICE


def seed_everything(seed=42):
    # 1. Set Python built-in random seed
    random.seed(seed)
    
    # 2. Set Python hash seed (dict/set iteration order)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. Set NumPy random seed
    np.random.seed(seed)
    
    # 4. Set PyTorch seeds
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if using multi-GPU
    
    # 5. Make CUDNN deterministic (might slightly impact performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Call this before dataset initialization and cross-validation splitting!
# seed_everything(42)

## Dataset

In [ ]:
from shared.normalization import ZScoreNormalizer

Datapoint = tuple[torch.Tensor, int, str]  # (patch tensor, class label, patient id)

class ElasticDataset(Dataset[Datapoint]):
    def __init__(
        self,
        mat_reader: MatReader,
        eff_fov_indices: list[int],
        train: bool = False,
        normalizer_override: ZScoreNormalizer | None = None
    ) -> None:

        self.mat_reader = mat_reader
        self.eff_fov_indices = eff_fov_indices
        self.train = train

        self.transform = A.Compose([
            # A.ElasticTransform(
            #     alpha=10,      
            #     sigma=6, 
            #     # alpha_affine=100 * 0.03, 
            #     border_mode=cv2.BORDER_REFLECT_101,
            #     p=1.0,
            # ),
            A.RandomRotate90(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            ToTensorV2() # Converts back to (C, H, W) and makes it a Tensor
        ])

        if normalizer_override:
            self.normalizer = normalizer_override
        else:
            # self.normalizer = ZScoreNormalizer(mat_reader, list(range(mat_reader.get_num_fovs()))) # TODO: leakage
            self.normalizer = ZScoreNormalizer(mat_reader, eff_fov_indices) # TODO: leakage

    def __len__(self) -> int:
        return len(self.eff_fov_indices)

    def __getitem__(self, idx: int) -> Datapoint:
        eff_idx = self.eff_fov_indices[idx]

        image = self.mat_reader.images[eff_idx] # (C, H, W)
        # image = robust_minmax(image, axis=(1, 2)).astype(np.float32)
        
        if self.train:
            image = image.transpose(1, 2, 0) # (H, W, C) for albumentations
            image = self.transform(image=image)['image']
        else:
            image = torch.from_numpy(image).float()

        image = self.normalizer.normalize(image)
        
        class_label = self.mat_reader.class_labels[eff_idx]
        patient_id = self.mat_reader.patient_ids[eff_idx]

        return image, class_label, patient_id


## Model

In [ ]:
import pretrained_microscopy_models as pmm
import torch.utils.model_zoo as model_zoo
from torchvision import models

class CustomModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        # weights = models.ResNet18_Weights.DEFAULT
        # model = models.resnet18(weights=weights)
        
        model = models.resnet50(weights=None)
        url = pmm.util.get_pretrained_microscopynet_url("resnet50", "micronet")
        model.load_state_dict(model_zoo.load_url(url, map_location=DEVICE))
        
        in_features = model.fc.in_features
        num_classes = len(CLASS_NAMES)
        # model.fc = nn.Sequential(  # type: ignore[assignment]
        #     nn.BatchNorm1d(in_features), # TODO: this is similar to StandardScaler step in SVM
        #     nn.Dropout(p=0.5),
        #     nn.Linear(in_features, num_classes),
        # )

        model.fc = nn.Sequential(
            nn.BatchNorm1d(in_features),
            
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            
            nn.Linear(128, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            
            nn.Linear(32, 1)
        )

        # model.fc = nn.Sequential(
        #     nn.BatchNorm1d(in_features),
        #     nn.Linear(in_features, 512),
        #     nn.ReLU(),
        #     nn.Dropout(p=0.5),
        #     nn.Linear(512, 1) 
        # )

        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True
        
        self.model = model
        self.fc = model.fc
        self.eval()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def train(self, mode: bool = True): # `mode` to match PyTorch signature
        # .eval() maps to .train(False)

        super().train(mode)

        if mode:
            # Push the whole backbone (including fc) into eval to freeze BatchNorm stats
            self.model.eval() # requires_grad is not enough: ".eval()" preserves BatchNorm
            
            # enables BatchNorm and Dropout of only the classifier
            self.fc.train()

# print(CustomModel()) # inspect architecture

## Feature Extraction

In [ ]:
from shared.utils import get_n_splits

mat_reader = MatReader("/home/student/lampe-cnn/full_images")

n_splits = get_n_splits(mat_reader)
n_splits = 5

## StratifiedGroupKFolds

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True)
# sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f"# splits: {n_splits}")

X, y, groups = mat_reader.images, mat_reader.class_labels, mat_reader.patient_ids

train_losses = [[] for _ in range(n_splits)]
train_targets = [0] * n_splits
train_preds = [0] * n_splits

val_losses = [[] for _ in range(n_splits)]
val_targets = [0] * n_splits
val_preds = [0] * n_splits
val_probs = [0] * n_splits

accuracies = [0] * n_splits
min_val_losses = [(10, 9999)] * n_splits # (epoch number, val loss)

for fold, (train_indices, val_indices) in enumerate(sgkf.split(X, y, groups=groups)):
    # minimum loss across epochs in the current fold
    min_val_loss = 9999
    
    train_dataset = ElasticDataset(mat_reader, eff_fov_indices=train_indices.tolist(), train=True)
    val_dataset = ElasticDataset(mat_reader, eff_fov_indices=val_indices.tolist(), train=False, normalizer_override=train_dataset.normalizer)

    # assumes class label ordering
    class_weights = 1.0 / np.bincount(mat_reader.class_labels[train_indices])
    sample_weights = class_weights[mat_reader.class_labels[train_indices]]
    
    sampler = WeightedRandomSampler(
        weights=sample_weights, 
        # num_samples=len(sample_weights), 
        num_samples=len(sample_weights) * 2,
        replacement=True
    )

    train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    model = CustomModel().to(DEVICE)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6
    )
    criterion = nn.BCEWithLogitsLoss()
    early_stopping = EarlyStopping(patience=20)

    num_epochs = 100
    unfreezing_guard = True

    for epoch in range(num_epochs):
        model.train()
        
        # epoch-level across all batches
        running_train_loss = 0.0
        epoch_train_targets = []
        epoch_train_preds = []

        custom_threshold = 0.5
        
        for batch in train_loader:
            optimizer.zero_grad() # gradient accumulates from start of epoch without this
            images, class_labels, patient_ids = batch
            # print(np.unique(class_labels, return_counts=True))
    
            inputs = images.to(DEVICE)
            logits = model(inputs).squeeze(1) # [batch_size, 1] -> [batch_size]
            # preds = (logits > 0).long()
            
            probs = torch.sigmoid(logits)
            preds = (probs > custom_threshold).long()
            
            epoch_train_targets.extend(class_labels.tolist())
            epoch_train_preds.extend(preds.tolist())

            targets = class_labels.to(DEVICE).float()
            loss = criterion(logits, targets)
            running_train_loss += loss.item() * images.size(0)
            
            loss.backward()
            optimizer.step()
    
        train_loss = running_train_loss / len(train_dataset)
        train_losses[fold].append(train_loss) # stores losses across all epochs

        with torch.no_grad():
            model.eval()
            
            running_val_loss = 0.0 # across batches
            epoch_val_targets = []
            epoch_val_preds = []
            epoch_val_probs = [] # ROC AUC
            
            for batch in val_loader:
                # NOTE: DataLoader converts class_labels: np.ndarray --> torch.Tensor
                images, class_labels, patient_ids = batch
                
                inputs = images.to(DEVICE)
                logits = model(inputs).squeeze(1)
                # preds = (logits > 0).long()

                probs = torch.sigmoid(logits)
                preds = (probs > custom_threshold).long()
                
                epoch_val_targets.extend(class_labels.tolist())
                epoch_val_preds.extend(preds.tolist())
                epoch_val_probs.extend(probs.tolist())

                targets = class_labels.to(DEVICE).float()
                loss = criterion(logits, targets)
                running_val_loss += loss.item() * images.size(0)
                
            val_loss = running_val_loss / len(val_dataset)
            val_losses[fold].append(val_loss) # stores losses across all epochs

            acc = float(accuracy_score(epoch_val_targets, epoch_val_preds))

            # epoch with lowest validation loss
            if val_loss < min_val_loss:
                min_val_loss = val_loss
                min_val_losses[fold] = (epoch, val_loss) # for matplotlib .scatter()

                # targets and preds for this fold
                train_targets[fold] = epoch_train_targets
                train_preds[fold] = epoch_train_preds
                val_targets[fold] = epoch_val_targets
                val_preds[fold] = epoch_val_preds

                # accuracy for this fold
                accuracies[fold] = acc

                # probablities (for ROC AUC)
                val_probs[fold] = epoch_val_probs

        
        scheduler.step(val_loss)
        early_stopping(val_loss)
        
        if early_stopping.early_stop:
            if unfreezing_guard:
                unfreezing_guard = False
                print(f"(Fold {fold + 1}): Unfreezing layer4 of ResNet50...")
                layer4_params = model.model.layer4.parameters()
                for param in layer4_params:
                    param.requires_grad = True
                optimizer.add_param_group({
                    'params': layer4_params,
                    'lr': 1e-5 # tinier LR than fc head
                })
                early_stopping = EarlyStopping(patience=20) # reset EarlyStopping counters
            else:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break

        print(
            (
                f"- Fold {fold + 1}/{n_splits}"
                f"- Epoch {epoch + 1}/{num_epochs}"
                f"- Train Loss: {train_loss:.4f}"
                f"- Val Loss: {val_loss:.4f}"
                f"- Accuracy: {acc:.4f}"
            )
        )

        

## Loss Curves, ROC AUC

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(2, n_splits, figsize=(5 * n_splits, 10))

# fig_title = "Report"
# fig.suptitle(fig_title)

# ax[0].imshow(img)
# ax[0].set_title('Original RGB')
# ax[0].axis('off')

auc_scores = []

for i in range(n_splits):
    # loss curves
    ax[0, i].plot(train_losses[i], label="Train Loss")
    ax[0, i].plot(val_losses[i], label="Val Loss")
    ax[0, i].scatter(*min_val_losses[i], color='red', s=100, zorder=5, label='Snapshot')
    ax[0, i].set_xlabel("Epoch")
    ax[0, i].set_ylabel("Loss")
    ax[0, i].set_title(f"Fold {i + 1}")
    ax[0, i].legend()

    # ROC AUC
    auc_score = roc_auc_score(val_targets[i], val_probs[i])
    auc_scores.append(auc_score)
    fpr, tpr, thresholds = roc_curve(val_targets[i], val_probs[i])
    ax[1, i].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc_score:.3f})')
    ax[1, i].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Random guessing line
    ax[1, i].set_xlim([0.0, 1.0])
    ax[1, i].set_ylim([0.0, 1.05])
    ax[1, i].set_xlabel('False Positive Rate')
    ax[1, i].set_ylabel('True Positive Rate')
    ax[1, i].set_title('Receiver Operating Characteristic')
    ax[1, i].legend(loc="lower right")


## Performance Metrics

In [ ]:
for fold in range(n_splits):
    cm = confusion_matrix(train_targets[fold], train_preds[fold])
    print(f"Train Confusion Matrix (Fold {fold+1}):")
    header = "          " + "  ".join(f"{name:>10}" for name in CLASS_NAMES)
    print(header)
    for i, row in enumerate(cm):
        row_str = "  ".join(f"{v:>10}" for v in row)
        print(f"{CLASS_NAMES[i]:>10}  {row_str}")

In [ ]:
for fold in range(n_splits):
    cm = confusion_matrix(val_targets[fold], val_preds[fold])
    print(f"Validation Confusion Matrix (Fold {fold+1} - {accuracies[fold] * 100:.2f}%):")
    header = "          " + "  ".join(f"{name:>10}" for name in CLASS_NAMES)
    print(header)
    for i, row in enumerate(cm):
        row_str = "  ".join(f"{v:>10}" for v in row)
        print(f"{CLASS_NAMES[i]:>10}  {row_str}")


## Report

In [ ]:
print("\n--- Final Results ---")
print(
    f"Mean CV Accuracy: {np.mean(accuracies) * 100:.2f}% ± {np.std(accuracies) * 100:.2f}%"
)
print(
    f"Mean CV AUC: {np.mean(auc_scores) * 100:.2f}% ± {np.std(auc_scores) * 100:.2f}%"
)

print("\nGlobal Classification Report:")
print(classification_report(np.concatenate(val_targets), np.concatenate(val_preds), target_names=CLASS_NAMES)) # must take only from last epochs

print(f"ACC: {accuracies}")
print(f"AUC: {auc_scores}")

## Grad-CAM

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

from shared.utils import robust_minmax

model = CustomModel()
model.load_state_dict(torch.load("/Users/james/GitHub/lampe/docs/results/model.pth"), weights_only=True)
model.eval()

cam = GradCAM(model=model.model, target_layers=target_layers)
target_layers = [model.model.layer4[-1]]
targets = [BinaryClassifierOutputTarget(1)]

In [ ]:
dataset = ElasticDataset(mat_reader, eff_fov_indices=list(range(mat_reader.get_num_fovs())), train=False)

image, class_label, patient_id = dataset[130]

grayscale_cam = cam(input_tensor=image.unsqueeze(0), targets=targets)

visualization = show_cam_on_image(robust_minmax(image.permute(1, 2, 0)), grayscale_cam[0, :], use_rgb=True)

logit = cam.outputs
confidence = torch.sigmoid(logit.squeeze())
print(confidence.item(), (confidence > 0.5).item())

fig, ax = plt.subplots(1, figsize=(8, 8))
ax.imshow(visualization)
ax.axis('off')